# Случайный лес: идея и практика

В этом ноутбуке мы разберёмся, как из отдельных решающих деревьев собрать более мощную модель — **случайный лес**.

**Цели ноутбука:**
- Понять, что такое **ансамблирование** и почему оно помогает.
- Разобрать, как именно устроен **Random Forest** (бутстрэп, случайный выбор признаков).
- Попробовать `RandomForestClassifier` из `scikit-learn` на синтетическом датасете.

---

**Содержание:** Ансамблирование → Устройство Random Forest → Данные → Обучение и оценка.


## От одиночного дерева к ансамблю

Одиночное дерево:
- легко **переобучается**, если не ограничивать глубину;
- обладает высокой **дисперсией** (чувствительно к шуму в данных и к случайному разбиению на train/test).

Идея ансамблирования:
- обучаем **несколько моделей** (деревьев) на слегка отличающихся выборках;
- объединяем их предсказания (например, голосованием);
- дисперсия сильно падает, а смещение (bias) часто остаётся примерно тем же.

Случайный лес — это ансамбль из многих деревьев, каждое из которых строится **на случайной подвыборке объектов и признаков**.


## Как строится случайный лес

Для каждого дерева в лесу:
1. Делаем **бутстрэп**: выбираем из обучающей выборки объекты с возвращением (т.е. один и тот же объект может попасть несколько раз, а какой‑то — не попасть).
2. На каждом разбиении внутри дерева выбираем **случайное подмножество признаков** и ищем лучший сплит только по ним.

На этапе предсказания:
- каждое дерево отдаёт свой класс (в классификации);
- итоговый прогноз — это **большинство голосов**.

Такая процедура уменьшает корреляцию между деревьями и обеспечивает хорошее качество из "шумных" базовых моделей.


In [7]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_moons(n_samples=400, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_train.shape, X_test.shape


((280, 2), (120, 2))

## Обучаем `RandomForestClassifier`

Попробуем построить лес с разным числом деревьев и ограничениями по глубине. Обратим внимание:
- как растёт качество с увеличением `n_estimators`;
- как влияет `max_depth` (сильное ограничение уменьшает переобучение, но может увеличить смещение).


In [8]:
def evaluate_forest(n_estimators: int, max_depth: int | None = None, random_state: int = 42):
    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state,
        n_jobs=-1,
    )
    clf.fit(X_train, y_train)
    y_pred_train = clf.predict(X_train)
    y_pred_test = clf.predict(X_test)
    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    print(f"n_estimators={n_estimators:3d}, max_depth={str(max_depth):>4} | ", end="")
    print(f"train_acc={train_acc:.3f}, test_acc={test_acc:.3f}")
    return clf

for n_estimators in [1, 5, 20, 100]:
    _ = evaluate_forest(n_estimators=n_estimators, max_depth=None)


n_estimators=  1, max_depth=None | train_acc=0.950, test_acc=0.900
n_estimators=  5, max_depth=None | train_acc=0.986, test_acc=0.958
n_estimators= 20, max_depth=None | train_acc=1.000, test_acc=0.942
n_estimators=100, max_depth=None | train_acc=1.000, test_acc=0.942


## Важность признаков в Random Forest

Случайный лес умеет оценивать, насколько каждый признак важен для итогового решения. Идея грубо следующая:
- если признак часто используется в разбиениях, которые сильно улучшают критерий (уменьшают Джини/энтропию), то его **важность** считается высокой.

В `sklearn` важности хранятся в атрибуте `feature_importances_` у обученного `RandomForestClassifier`.


In [9]:
clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)

importances = clf.feature_importances_
for idx, imp in enumerate(importances):
    print(f"Признак {idx}: важность = {imp:.3f}")


Признак 0: важность = 0.439
Признак 1: важность = 0.561


## Выводы

- Случайный лес снижает дисперсию за счёт бутстрэпа и случайного выбора признаков при каждом разбиении; итог — голосование деревьев.
- Ансамбль устойчивее одиночного дерева к переобучению и даёт полезные оценки важности признаков (`feature_importances_`).